In [1]:
# get openai model
# load faq dataset
# get 50 samples
# generate eval dataset for faq
# persist to data/eval

In [2]:
import sys
sys.path.append("../")
import json
import pandas as pd

from models.generation import openai_generate
from ingestion.faq.faq_heading_chunker import parse_faq_docx

In [3]:
# load faq dataset
docs = parse_faq_docx("../data/processed/faq.docx")
docs[:100]

[{'section_id': 1,
  'header': 'How to register with an email',
  'content': 'Open the\xa0registration page. Choose whether you want to hold a\xa0regular account or a business account.\nEnter your email address and create a password.\nDetermine whether you are over 18 years old. If not, provide your exact date of birth. You will hold a\xa0Junior account\xa0which will be transformed into a regular account once you turn 18.\nAccept the\xa0HopShop Terms & Conditions\xa0and click [register].\nYou will receive a confirmation email. Open your mailbox and click (confirm registration) to complete the registration process.\nOnce you complete the registration, we will redirect you to your new HopShop account. We will give you a\xa0default\xa0username ― you can change it in the\xa0Account details\xa0tab.'},
 {'section_id': 2,
  'header': 'How to register with Google or Facebook account',
  'content': 'Open the\xa0sign in page.\nSelect signing in through Facebook or Google.\nSign in to your Facebo

In [4]:
def make_prompt(header, content, section_id):
    return f"""
  You are generating evaluation examples for a FAQ retrieval system.

  Your task is to create exactly one user question based on the provided FAQ data:
  - Header: the FAQ title or topic
  - Content: the reference answer

  Requirements:
  - Generate a natural user-style question that could realistically be asked to retrieve this FAQ answer.
  - The question must be fully grounded in the Header and Content.
  - The question can also be a paraphrased version of the original user intent while preserving the same meaning.
  - Do not invent facts that are not supported by the input.
  - The question should be specific enough to match the Content.
  - Prefer concise, realistic phrasing.
  - Generate only one question.

  Return the output as valid JSON with exactly this schema:
  {{"question": "<generated question>", "reference_answer": "<Content>", "relevant_docs": [<section_id>]}}

  Input:
  Header: {header}
  Content: {content}
  Section_Id: {section_id}
    """

In [5]:
rows = []
for ob in docs[:50]:
    prompt = make_prompt(ob['header'], ob['content'], ob["section_id"])
    resp = openai_generate(prompt)
    rows.append(resp)

In [6]:
with open("../data/eval/faq_evaluation_dataset.jsonl", "w", encoding="utf-8") as f:
    for row in rows:
        if isinstance(row, str):
            row = json.loads(row)

        f.write(json.dumps(row, ensure_ascii=False) + "\n")

In [7]:
df = pd.read_json("../data/eval/faq_evaluation_dataset.jsonl", lines=True)
df.head(5)

,question,reference_answer,relevant_docs
0,How do I register for a HopShop account using ...,Open the registration page. Choose whether you...,[1]
1,How can I sign up using my Google or Facebook ...,Open the sign in page.\nSelect signing in thro...,[2]
2,How can I enable signing in with my email addr...,If you want to be able to sign in to your acco...,[3]
3,How do I register for a HopShop account using ...,Open the registration page. Select whether you...,[4]
4,How can I download my personal data from my Ho...,"On the Copies of your data page, you can downl...",[5]
